In [3]:
import os
from dotenv import load_dotenv

load_dotenv(override=True)

True

<b><b> MODEL CALL<b><b>

In [1]:
from google import genai

client = genai.Client()
model="gemini-3.1-flash-lite"



<b>Structured Output</b>

In [17]:
from typing import List, Literal
from pydantic import BaseModel, Field

class ResearchOutput(BaseModel):
    task_type: Literal[
        "concept_explanation",
        "paper_summary",
        "method_explanation",
        "research_gap_analysis",
        "literature_overview",
        "general_research_help",
    ] = Field(
        description="What kind of research help the user is asking for."
    )

    topic: str = Field(
        description="The main topic, paper, concept, or research area in the user's query."
    )

    goal: str = Field(
        description="A one-line description of what the user wants to learn, understand, or produce."
    )

    key_points: List[str] = Field(
        description="The main answer in simple, beginner-friendly bullet points."
    )

    examples: List[str] = Field(
        description="Simple examples or analogies that make the explanation easier to understand."
    )

    follow_up_suggestions: List[str] = Field(
        description="Helpful next steps, follow-up questions, or related things the user may want to explore."
    )



<b>System Prompt</b>


In [26]:
import yaml
from pathlib import Path

_PROMPTS = yaml.safe_load(Path("prompt.yml").read_text(encoding="utf-8"))
RESEARCH_SYSTEM = _PROMPTS["RESEARCH_SYSTEM"]



<b>Cost & Pricing</b>

In [27]:
IN_PRICE_PER_MTOK = 0.25
OUT_PRICE_PER_MTOK = 1.50

totals = {"input": 0, "output": 0}


def track_cost(usage) -> None:
    
    totals["input"] += usage.prompt_token_count or 0
    totals["output"] += usage.candidates_token_count or 0


def usage_report() -> str:
    """Summarise tokens used and the estimated cost so far."""
    cost = (
        totals["input"] / 1e6 * IN_PRICE_PER_MTOK
        + totals["output"] / 1e6 * OUT_PRICE_PER_MTOK
    )
    return f"{totals['input']} in + {totals['output']} out tokens = ~${cost:.4f}"

<b>Reasearch Asssitant</b>

In [28]:
def research_assistant(message: str) -> ResearchOutput:
    resp = client.models.generate_content(
        model=model,
        contents=message,
        config={
            "system_instruction": RESEARCH_SYSTEM,
            "response_mime_type": "application/json",
            "response_schema": ResearchOutput,
        },
    )
    track_cost(resp.usage_metadata)   
    return resp.parsed                

<b>Draft reply</b>

In [29]:
def draft_reply(result: ResearchOutput) -> None:
    print("TASK TYPE :", result.task_type)
    print("TOPIC     :", result.topic)
    print("GOAL      :", result.goal)

    print("\nKEY POINTS:")
    for point in result.key_points:
        print(f" - {point}")

    print("\nEXAMPLES:")
    if result.examples:
        for ex in result.examples:
            print(f" - {ex}")
    else:
        print(" - None")

    print("\nFOLLOW-UP SUGGESTIONS:")
    if result.follow_up_suggestions:
        for item in result.follow_up_suggestions:
            print(f" - {item}")
    else:
        print(" - None")

In [31]:
def main() -> None:
    samples = [
        "Explain overfitting in machine learning in simple bullet points with an example.",
        
      
    ]

    for s in samples:
        result = research_assistant(s)

        print("-" * 70)
        print("MESSAGE :", s)
       # print("OUTPUT  :", result.model_dump())
        draft_reply(result)

      

    print("=" * 70)
    print("TOKENS  :", usage_report())


if __name__ == "__main__":
    main()

----------------------------------------------------------------------
MESSAGE : Explain overfitting in machine learning in simple bullet points with an example.
TASK TYPE : concept_explanation
TOPIC     : Overfitting in machine learning
GOAL      : Explain what overfitting is and how it affects model performance.

KEY POINTS:
 - Overfitting happens when a model learns the training data too well, including the noise and random fluctuations.
 - The model becomes overly complex and fails to generalize to new, unseen data.
 - It performs perfectly on data it has already seen but poorly on real-world test data.
 - It is often caused by having a model that is too complex relative to the amount of training data available.
 - Techniques like simplifying the model, using more training data, or regularization can help prevent it.

EXAMPLES:
 - Imagine a student who memorizes every single practice question and answer for an exam instead of understanding the underlying concepts; when the actual e